# Representation Demystifies Flat Minima
## Notebook 49: Function-Space Robustness

Notebook 43 showed that a SAM-style parameter-space penalty can change under reparameterization.

This notebook asks the next question:

> If parameter-space sharpness is representation-dependent, what quantity survives?

We compare perturbations in parameter coordinates with perturbations in function space.

## Output files

Running this notebook creates:

- `figures/49_parameter_radius_function_change.png`
- `figures/49_function_space_prediction_drift.png`
- `figures/49_parameter_vs_function_drift.png`
- `figures/49_invariant_hierarchy.png`
- `figures/49_representation_vs_computation.png`
- `results/49_function_space_robustness_summary.csv`
- `results/49_function_space_robustness_summary.json`
- `downloads/49_function_space_robustness_outputs.zip`

In [ ]:
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIGURES = ROOT / "figures"
RESULTS = ROOT / "results"
DOWNLOADS = ROOT / "downloads"

for path in [FIGURES, RESULTS, DOWNLOADS]:
    path.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("FIGURES:", FIGURES)
print("RESULTS:", RESULTS)
print("DOWNLOADS:", DOWNLOADS)

## 1. Model and reparameterization

Use the same simple model:

\[
f(x;w)=wx
\]

with:

\[
w=su
\]

At:

\[
u^*=\frac{1}{s}
\]

all representations compute the same function:

\[
f(x)=x
\]

In [ ]:
x = np.linspace(-1, 1, 800)
y = x

scales = np.array([0.25, 0.5, 1, 2, 4, 8])

delta_u = 0.05
delta_w = 0.05

rows = []

for s in scales:
    u_star = 1 / s
    w_star = s * u_star

    # Fixed parameter-space perturbation in u.
    w_from_delta_u = s * (u_star + delta_u)
    function_change_from_delta_u = abs(w_from_delta_u - w_star)
    pred_drift_from_delta_u = np.mean(np.abs((w_from_delta_u * x) - (w_star * x)))

    # Fixed function-space perturbation in effective w.
    w_from_delta_w = w_star + delta_w
    equivalent_delta_u = delta_w / s
    pred_drift_from_delta_w = np.mean(np.abs((w_from_delta_w * x) - (w_star * x)))

    rows.append({
        "scale_s": float(s),
        "u_star": float(u_star),
        "w_star": float(w_star),
        "fixed_delta_u": float(delta_u),
        "effective_delta_w_from_delta_u": float(function_change_from_delta_u),
        "prediction_drift_from_delta_u": float(pred_drift_from_delta_u),
        "fixed_delta_w": float(delta_w),
        "equivalent_delta_u_for_fixed_delta_w": float(equivalent_delta_u),
        "prediction_drift_from_delta_w": float(pred_drift_from_delta_w),
    })

results = pd.DataFrame(rows)
results

## 2. Same parameter radius, different function radius

A fixed coordinate perturbation:

\[
|\Delta u| = 0.05
\]

produces:

\[
|\Delta w| = s|\Delta u|
\]

Therefore the same parameter-space radius is not the same function-space radius.

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    results["scale_s"],
    results["effective_delta_w_from_delta_u"],
    marker="o"
)

plt.xscale("log")
plt.yscale("log")
plt.xlabel("representation scale s")
plt.ylabel("effective function-space perturbation |Δw|")
plt.title("Same parameter radius produces different function changes")
plt.grid(True, which="both")

fig_path = FIGURES / "49_parameter_radius_function_change.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()

print("Saved:", fig_path)

## 3. Same function-space perturbation

Instead of fixing \(\Delta u\), fix the effective function-space perturbation:

\[
|\Delta w| = 0.05
\]

Then every representation has the same output drift.

In [ ]:
plt.figure(figsize=(8, 5))

for s in scales:
    u_star = 1 / s
    w_star = s * u_star

    y_base = w_star * x
    y_perturbed = (w_star + delta_w) * x
    drift = y_perturbed - y_base

    plt.plot(x, drift, label=f"s={s}")

plt.xlabel("x")
plt.ylabel("prediction drift")
plt.title("Fixed function-space perturbation gives identical drift")
plt.legend()
plt.grid(True)

fig_path = FIGURES / "49_function_space_prediction_drift.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()

print("Saved:", fig_path)

## 4. Parameter-space drift versus function-space drift

Compare mean absolute prediction drift under:

- fixed parameter-space radius
- fixed function-space radius

Only the function-space perturbation remains stable across representations.

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    results["scale_s"],
    results["prediction_drift_from_delta_u"],
    marker="o",
    label="fixed parameter-space radius"
)

plt.plot(
    results["scale_s"],
    results["prediction_drift_from_delta_w"],
    marker="o",
    label="fixed function-space radius"
)

plt.xscale("log")
plt.yscale("log")
plt.xlabel("representation scale s")
plt.ylabel("mean absolute prediction drift")
plt.title("Function-space robustness is representation-stable")
plt.legend()
plt.grid(True, which="both")

fig_path = FIGURES / "49_parameter_vs_function_drift.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()

print("Saved:", fig_path)

## 5. Invariant hierarchy

This notebook updates the repo's invariant checklist.

Parameter-space quantities are useful, but they do not automatically survive equivalent reparameterizations.

In [ ]:
hierarchy = pd.DataFrame([
    {"quantity": "Parameter coordinates", "level": "representation", "invariant": False},
    {"quantity": "Parameter norm", "level": "representation", "invariant": False},
    {"quantity": "Hessian sharpness", "level": "geometry", "invariant": False},
    {"quantity": "SAM parameter penalty", "level": "geometry / objective", "invariant": False},
    {"quantity": "Function", "level": "computation", "invariant": True},
    {"quantity": "Predictions", "level": "computation", "invariant": True},
    {"quantity": "Function-space perturbation", "level": "computation", "invariant": True},
    {"quantity": "Prediction drift under fixed Δw", "level": "computation", "invariant": True},
])

hierarchy

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.axis("off")

display_df = hierarchy.copy()
display_df["invariant"] = display_df["invariant"].map({True: "yes", False: "no"})

table = ax.table(
    cellText=display_df.values,
    colLabels=["Quantity", "Level", "Invariant?"],
    loc="center",
    cellLoc="left",
    colLoc="left"
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.55)

ax.set_title("Which quantities survive representation changes?")

fig_path = FIGURES / "49_invariant_hierarchy.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()

print("Saved:", fig_path)

## 6. Representation versus computation

This is the conceptual shift:

- parameter-space flatness belongs to representation geometry
- prediction drift belongs closer to computation

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.axis("off")

ax.text(0.25, 0.82, "Representation", ha="center", fontsize=18)
ax.text(0.25, 0.58, "Coordinates", ha="center", fontsize=16)
ax.text(0.25, 0.38, "Sharpness", ha="center", fontsize=16)
ax.text(0.25, 0.18, "SAM penalty", ha="center", fontsize=16)

ax.text(0.75, 0.82, "Computation", ha="center", fontsize=18)
ax.text(0.75, 0.58, "Function", ha="center", fontsize=16)
ax.text(0.75, 0.38, "Predictions", ha="center", fontsize=16)
ax.text(0.75, 0.18, "Robustness", ha="center", fontsize=16)

ax.text(
    0.5,
    0.02,
    "Robustness should be measured where the computation lives.",
    ha="center",
    fontsize=15
)

fig_path = FIGURES / "49_representation_vs_computation.png"
plt.savefig(fig_path, bbox_inches="tight", dpi=180)
plt.show()

print("Saved:", fig_path)

## 7. Save results

In [ ]:
csv_path = RESULTS / "49_function_space_robustness_summary.csv"
json_path = RESULTS / "49_function_space_robustness_summary.json"

results.to_csv(csv_path, index=False)

summary = {
    "title": "Function-Space Robustness",
    "fixed_delta_u": float(delta_u),
    "fixed_delta_w": float(delta_w),
    "min_prediction_drift_from_delta_u": float(results["prediction_drift_from_delta_u"].min()),
    "max_prediction_drift_from_delta_u": float(results["prediction_drift_from_delta_u"].max()),
    "prediction_drift_from_fixed_delta_w": float(results["prediction_drift_from_delta_w"].iloc[0]),
    "max_drift_ratio_parameter_to_function": float(
        results["prediction_drift_from_delta_u"].max() /
        results["prediction_drift_from_delta_w"].iloc[0]
    ),
    "engineering_statement": (
        "Generalization should be explained using quantities that survive "
        "equivalent representations. Function-space robustness is more "
        "representation-stable than parameter-space sharpness."
    )
}

with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)

print("Saved:", csv_path)
print("Saved:", json_path)
print(summary)

## Notebook takeaway

Fixed parameter-space perturbations do not mean fixed functional perturbations.

Fixed function-space perturbations do.

Therefore:

> Generalization should be explained using quantities that survive equivalent representations.
>
> Function-space robustness is more representation-stable than parameter-space sharpness.

In [ ]:
output_files = [
    FIGURES / "49_parameter_radius_function_change.png",
    FIGURES / "49_function_space_prediction_drift.png",
    FIGURES / "49_parameter_vs_function_drift.png",
    FIGURES / "49_invariant_hierarchy.png",
    FIGURES / "49_representation_vs_computation.png",
    RESULTS / "49_function_space_robustness_summary.csv",
    RESULTS / "49_function_space_robustness_summary.json",
]

zip_path = DOWNLOADS / "49_function_space_robustness_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in output_files:
        if f.exists():
            z.write(f, arcname=f.relative_to(ROOT))

print("Created:", zip_path)
print("\nIncluded files:")
for f in output_files:
    print("-", f.relative_to(ROOT), "exists=" + str(f.exists()))

## Download outputs

In Colab, run the next cell to download the ZIP directly.

In local Jupyter, open:

`downloads/49_function_space_robustness_outputs.zip`

In [ ]:
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display

    display(FileLink(str(zip_path)))
    for f in output_files:
        if f.exists():
            display(FileLink(str(f)))